In [3]:
import pandas as pd
import pyodbc
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

In [4]:
import pyodbc
print(pyodbc.drivers())

['SQL Server', 'SQL Server Native Client RDA 11.0', 'ODBC Driver 17 for SQL Server', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)']


In [8]:
import pyodbc
import pandas as pd

def fetchdata():
    conn_str = (
        "DRIVER={ODBC Driver 17 for SQL Server};"
        "SERVER=DESKTOP-UG1R0GK\\SQLEXPRESS;"
        "DATABASE=MarketingAnalytics;"
        "Trusted_Connection=yes;"
    )

    conn = pyodbc.connect(conn_str)

    query = """
    SELECT 
        ReviewID, 
        CustomerID, 
        ProductID, 
        ReviewDate, 
        Rating, 
        ReviewText
    FROM customer_reviews
    """

    data = pd.read_sql(query, conn)

    conn.close()
    return data

In [9]:
# Fetch Customer Review Data from SQL database 
cus_review_data =  fetchdata()

C:\Users\AL-MALAK\AppData\Local\Temp\ipykernel_9764\605421905.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data = pd.read_sql(query, conn)


In [11]:
cus_review_data.head()

,ReviewID,CustomerID,ProductID,ReviewDate,Rating,ReviewText
0,1,77,18,2023-12-23,3,"Average experience, nothing special."
1,2,80,19,2024-12-25,5,The quality is top-notch.
2,3,50,13,2025-01-26,4,Five stars for the quick delivery.
3,4,78,15,2025-04-21,3,"Good quality, but could be cheaper."
4,5,64,2,2023-07-16,3,"Average experience, nothing special."


In [12]:
## Intialise the Setntiment intensity analyzer
sentinaly = SentimentIntensityAnalyzer()

In [21]:
## function identify to calculate sentiment score using Vader
def sentiment_calculate(review):
    ## Get sentiiment score for the reviewtext
    sentiment = sentinaly.polarity_scores(review)
    
    return sentiment['compound'] ##Return compound score that is between -1 to 1 means most negative to most positive 

In [15]:
def categorize_sentiment(score, rating):

    if score > 0.05:
        if rating >= 4:
            return 'Positive'
        elif rating == 3:
            return 'Mixed Positive'
        else:
            return 'Mixed Negative'

    elif score < -0.05:
        if rating <= 2:
            return 'Negative'
        elif rating == 3:
            return 'Mixed Negative'
        else:
            return 'Mixed Positive'

    rating_map = {
        5: 'Positive',
        4: 'Positive',
        3: 'Neutral',
        2: 'Negative',
        1: 'Negative'
    }

    return rating_map.get(rating, 'Neutral')

In [17]:
def bucket_sentiment(score):
    if score >= 0.5:
        return '0.5 to 1.0'  # Strongly positive sentiment

    elif 0.0 <= score < 0.5:
        return '0.0 to 0.49'  # Mildly positive sentiment

    elif -0.5 <= score < 0.0:
        return '-0.49 to 0.0'  # Mildly negative sentiment

    else:
        return '-1.0 to -0.5'  # Strongly negative sentiment

In [18]:
cus_review_data.head()

,ReviewID,CustomerID,ProductID,ReviewDate,Rating,ReviewText
0,1,77,18,2023-12-23,3,"Average experience, nothing special."
1,2,80,19,2024-12-25,5,The quality is top-notch.
2,3,50,13,2025-01-26,4,Five stars for the quick delivery.
3,4,78,15,2025-04-21,3,"Good quality, but could be cheaper."
4,5,64,2,2023-07-16,3,"Average experience, nothing special."


In [22]:
cus_review_data['SentimentScore'] = cus_review_data['ReviewText'].apply(sentiment_calculate)

In [23]:
cus_review_data.head()

,ReviewID,CustomerID,ProductID,ReviewDate,Rating,ReviewText,SentimentScore
0,1,77,18,2023-12-23,3,"Average experience, nothing special.",-0.3089
1,2,80,19,2024-12-25,5,The quality is top-notch.,0.0000
2,3,50,13,2025-01-26,4,Five stars for the quick delivery.,0.0000
3,4,78,15,2025-04-21,3,"Good quality, but could be cheaper.",0.2382
4,5,64,2,2023-07-16,3,"Average experience, nothing special.",-0.3089


In [24]:
cus_review_data['CategoryofSentiment'] = cus_review_data.apply(
    lambda row: categorize_sentiment(row['SentimentScore'],row['Rating']), axis =1 )

In [25]:
cus_review_data

,ReviewID,CustomerID,ProductID,ReviewDate,Rating,ReviewText,SentimentScore,CategoryofSentiment
0,1,77,18,2023-12-23,3,"Average experience, nothing special.",-0.3089,Mixed Negative
1,2,80,19,2024-12-25,5,The quality is top-notch.,0.0000,Positive
2,3,50,13,2025-01-26,4,Five stars for the quick delivery.,0.0000,Positive
3,4,78,15,2025-04-21,3,"Good quality, but could be cheaper.",0.2382,Mixed Positive
4,5,64,2,2023-07-16,3,"Average experience, nothing special.",-0.3089,Mixed Negative
...,...,...,...,...,...,...,...,...
1358,1359,28,4,2023-05-25,3,Not worth the money.,-0.1695,Mixed Negative
1359,1360,58,12,2023-11-13,2,"Average experience, nothing special.",-0.3089,Negative
1360,1361,96,15,2023-03-07,5,Customer support was very helpful.,0.6997,Positive
1361,1362,99,2,2025-12-03,1,Product did not meet my expectations.,0.0000,Negative


In [26]:
cus_review_data['SentimentBucket'] = cus_review_data['SentimentScore'].apply(bucket_sentiment)

In [27]:
cus_review_data.head()

,ReviewID,CustomerID,ProductID,ReviewDate,Rating,ReviewText,SentimentScore,CategoryofSentiment,SentimentBucket
0,1,77,18,2023-12-23,3,"Average experience, nothing special.",-0.3089,Mixed Negative,-0.49 to 0.0
1,2,80,19,2024-12-25,5,The quality is top-notch.,0.0000,Positive,0.0 to 0.49
2,3,50,13,2025-01-26,4,Five stars for the quick delivery.,0.0000,Positive,0.0 to 0.49
3,4,78,15,2025-04-21,3,"Good quality, but could be cheaper.",0.2382,Mixed Positive,0.0 to 0.49
4,5,64,2,2023-07-16,3,"Average experience, nothing special.",-0.3089,Mixed Negative,-0.49 to 0.0


In [28]:
cus_review_data.to_csv('fact_customer_reviewa_with_sentiment', index = False)